In [1]:
import polars as pl
import pandas as pd
import plotly.express as px
import pingouin as pg
from scipy.stats import mannwhitneyu

from itertools import chain
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from pyprojroot import here

root_dir = here()

In [2]:
construct_items = {
    "OA": [
        "OA1",
        # "OA2", # Killed due to low CR-A
        "OA3",
        "OA4",
    ],
    "PA": [
        "PA1",
        "PA2",
        "PA3",
        "PA4",
    ],
    "ITU": [
        "IUT1",
        "IUT2",
        "IUT3",
    ],
    "PRIV": [
        "PRIV1",
        "PRIV2",
        # "PRIV3", # Killed due to low CR-A (reversed)
    ],
    "RA": [
        "RA1",
        "RA2",
        "RA3",
    ],
    "RC": [
        "RC1",
        "RC2",
        "RC3",
    ],
}


required_cols = [
    "ATT_SC",
    "CTRL1",
    "GROUP",
    "IUT1",
    "IUT2",
    "IUT3",
    "MARK1",
    "MARK2",
    "mc1",
    "MC2",
    "OA1",
    "OA2",
    "OA3",
    "OA4",
    "PA1",
    "PA2",
    "PA3",
    "PA4",
    "PAKNOW",
    "PRIV1",
    "PRIV2",
    "PRIV3",
    "RA1",
    "RA2",
    "RA3",
    "RC1",
    "RC2",
    "RC3",
    "RD01_CP",
    "RD02_CP",
    "RD02",
    "RISK",
]


In [3]:
df_raw = pl.from_pandas(
    pd.read_csv(
        root_dir / "data" / "soci" / "0-300" / "cleaned.csv", encoding="ISO-8859-1"
    )
)

In [4]:
drop_items = [
    "OA2",
    "PRIV3",
]

In [5]:
df = (
    df_raw.filter(
        pl.col("FT02_01") != "MiriamTest",
    )
    .drop_nulls(subset=required_cols)
    .drop(drop_items)
    .with_columns(ai_reporting=pl.col("GROUP").is_in([1, 2]).cast(pl.Int8))
    .rename({
        "PAKNOW": "control_PAKNOW"
    })
    .to_pandas()
)

In [6]:
import plspm.config as c
from plspm.plspm import Plspm
from plspm.scheme import Scheme
from plspm.mode import Mode


In [7]:
structure = c.Structure()

In [8]:
structure.add_path(["AI_REPORTING"], ["RA", "RC"])
structure.add_path(["RA"], ["PA", "OA"])
structure.add_path(["RC"], ["PA", "OA"])
structure.add_path(["PA"], ["ITU"])
structure.add_path(["OA"], ["ITU"])

In [9]:
config = c.Config(structure.path(), scaled=False)


In [10]:
config.add_lv_with_columns_named("AI_REPORTING", Mode.B, df, "ai_reporting")

config.add_lv_with_columns_named("RA", Mode.A, df, "RA")
config.add_lv_with_columns_named("RC", Mode.B, df, "RC")
config.add_lv_with_columns_named("PA", Mode.B, df, "PA")
config.add_lv_with_columns_named("OA", Mode.B, df, "OA")
config.add_lv_with_columns_named("ITU", Mode.A, df, "IUT")


In [ ]:
plspm_calc = Plspm(
    df,
    config,
    Scheme.PATH,
    300,
    bootstrap=True,
    bootstrap_iterations=5_000,
    processes=10,
)

In [12]:
plspm_calc.inner_summary()

,type,r_squared,r_squared_adj,block_communality,mean_redundancy,ave
AI_REPORTING,Exogenous,0.000000,0.000000,1.000000,0.000000,NaN
ITU,Endogenous,0.171848,0.166253,0.938519,0.161283,0.938519
OA,Endogenous,0.106709,0.100673,0.376022,0.040125,NaN
PA,Endogenous,0.215278,0.209976,0.302129,0.065042,NaN
RA,Endogenous,0.024958,0.021675,0.709707,0.017713,0.709707
RC,Endogenous,0.000680,-0.002685,0.570010,0.000387,NaN


In [13]:
plspm_calc.path_coefficients()

,AI_REPORTING,RC,RA,OA,PA,ITU
AI_REPORTING,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
RC,0.026072,0.000000,0.000000,0.000000,0.000000,0.0
RA,0.157981,0.000000,0.000000,0.000000,0.000000,0.0
OA,0.000000,0.222079,0.171359,0.000000,0.000000,0.0
PA,0.000000,0.377813,0.164014,0.000000,0.000000,0.0
ITU,0.000000,0.000000,0.000000,0.141404,0.356991,0.0


In [14]:
plspm_calc.outer_model()

,weight,loading,communality,redundancy
IUT1,0.195093,0.979012,0.958464,1.647103e-01
IUT2,0.184630,0.958320,0.918378,1.578216e-01
IUT3,0.171903,0.968872,0.938714,1.613162e-01
OA1,0.741612,0.998079,0.996162,1.062992e-01
OA3,-0.032460,0.192955,0.037231,3.972929e-03
OA4,-0.013955,0.307688,0.094672,1.010233e-02
PA1,0.599719,0.958565,0.918847,1.978079e-01
PA2,-0.184413,-0.001250,0.000002,3.364376e-07
PA3,0.114890,0.410646,0.168630,3.630243e-02
PA4,0.002845,0.347905,0.121038,2.605690e-02


In [15]:
plspm_calc.inner_model()

,from,to,estimate,std error,t,p>|t|
index,,,,,,
AI_REPORTING -> RC,AI_REPORTING,RC,0.026072,0.058006,0.449475,6.534170e-01
AI_REPORTING -> RA,AI_REPORTING,RA,0.157981,0.057297,2.757214,6.190628e-03
RC -> OA,RC,OA,0.222079,0.059087,3.758511,2.058987e-04
RA -> OA,RA,OA,0.171359,0.059087,2.900125,4.009557e-03
RC -> PA,RC,PA,0.377813,0.055380,6.822197,5.058190e-11
RA -> PA,RA,PA,0.164014,0.055380,2.961614,3.308238e-03
OA -> ITU,OA,ITU,0.141404,0.054512,2.594021,9.958010e-03
PA -> ITU,PA,ITU,0.356991,0.054512,6.548883,2.559367e-10


In [16]:
plspm_calc.effects()

,from,to,direct,indirect,total
AI_REPORTING -> RC,AI_REPORTING,RC,0.026072,0.000000,0.026072
AI_REPORTING -> RA,AI_REPORTING,RA,0.157981,0.000000,0.157981
AI_REPORTING -> OA,AI_REPORTING,OA,0.000000,0.032862,0.032862
AI_REPORTING -> PA,AI_REPORTING,PA,0.000000,0.035761,0.035761
AI_REPORTING -> ITU,AI_REPORTING,ITU,0.000000,0.017413,0.017413
RC -> OA,RC,OA,0.222079,0.000000,0.222079
RC -> PA,RC,PA,0.377813,0.000000,0.377813
RC -> ITU,RC,ITU,0.000000,0.166279,0.166279
RA -> OA,RA,OA,0.171359,0.000000,0.171359
RA -> PA,RA,PA,0.164014,0.000000,0.164014
